In [1]:
import os
data_dir = '/home/devang/Projects/PilotCrew/Data-Science-Bench/tasks/task_06/data'
os.listdir(data_dir)

['accounts.csv', 'invoices.csv', 'tickets.csv', 'plan_history.csv']

In [2]:
import pandas as pd
plan_history = pd.read_csv(os.path.join(data_dir, 'plan_history.csv'))
invoices = pd.read_csv(os.path.join(data_dir, 'invoices.csv'))
print("plan_history columns:", plan_history.columns)
print(plan_history.head())
print("\ninvoices columns:", invoices.columns)
print(invoices.head())

plan_history columns: Index(['account_id', 'month', 'plan_tier', 'seats'], dtype='str')
  account_id       month plan_tier  seats
0        A01  2024-01-01   starter      6
1        A01  2024-02-01   starter      7
2        A01  2024-03-01   starter      8
3        A01  2024-04-01   starter      9
4        A01  2024-05-01    growth     10

invoices columns: Index(['invoice_id', 'account_id', 'month', 'issue_date', 'due_date',
       'paid_date', 'plan_tier', 'amount', 'credit_amount'],
      dtype='str')
  invoice_id account_id       month  issue_date    due_date   paid_date  \
0    INV0001        A01  2024-01-01  2024-01-05  2024-01-19  2024-01-06   
1    INV0002        A01  2024-02-01  2024-02-05  2024-02-19  2024-02-07   
2    INV0003        A01  2024-03-01  2024-03-05  2024-03-19  2024-03-08   
3    INV0004        A01  2024-04-01  2024-04-05  2024-04-19  2024-04-09   
4    INV0005        A01  2024-05-01  2024-05-05  2024-05-19  2024-05-10   

  plan_tier  amount  credit_amount  
0  

In [3]:
print(plan_history['plan_tier'].unique())
print(plan_history.groupby('account_id')['plan_tier'].nunique().value_counts())

<StringArray>
['starter', 'growth', 'enterprise']
Length: 3, dtype: str
plan_tier
2    8
1    4
Name: count, dtype: int64


In [4]:
# Check if any account downgraded
plan_order = {'starter': 1, 'growth': 2, 'enterprise': 3}
plan_history['tier_num'] = plan_history['plan_tier'].map(plan_order)

account_upgrades = plan_history.groupby('account_id')['tier_num'].agg(['min', 'max', 'first', 'last'])
account_upgrades['is_upgraded'] = account_upgrades['last'] > account_upgrades['first']
print(account_upgrades)

upgraded_accounts = account_upgrades[account_upgrades['is_upgraded']].index.tolist()
never_upgraded_accounts = account_upgrades[~account_upgrades['is_upgraded']].index.tolist()

print("Upgraded:", upgraded_accounts)
print("Never upgraded:", never_upgraded_accounts)

            min  max  first  last  is_upgraded
account_id                                    
A01           1    2      1     1        False
A02           1    2      1     1        False
A03           1    2      1     1        False
A04           1    2      1     1        False
A05           2    3      3     3        False
A06           2    3      2     2        False
A07           2    3      2     2        False
A08           2    3      2     2        False
A09           3    3      3     3        False
A10           3    3      3     3        False
A11           3    3      3     3        False
A12           3    3      3     3        False
Upgraded: []
Never upgraded: ['A01', 'A02', 'A03', 'A04', 'A05', 'A06', 'A07', 'A08', 'A09', 'A10', 'A11', 'A12']


In [5]:
invoices['net_amount'] = invoices['amount'] - invoices['credit_amount']

upgraded_invoices = invoices[invoices['account_id'].isin(upgraded_accounts)]
never_upgraded_invoices = invoices[invoices['account_id'].isin(never_upgraded_accounts)]

mean_upgraded = upgraded_invoices['net_amount'].mean()
mean_never_upgraded = never_upgraded_invoices['net_amount'].mean()

print("Mean upgraded:", mean_upgraded)
print("Mean never upgraded:", mean_never_upgraded)
print("Difference:", round(mean_upgraded - mean_never_upgraded, 2))

Mean upgraded: nan
Mean never upgraded: 483.7857142857143
Difference: nan


In [6]:
# Let's double check the definition of "upgraded accounts".
# An account is upgraded if it ever changed to a higher tier.
# What if it changed to a higher tier and then downgraded?
# The prompt says "upgraded accounts" and "never-upgraded accounts".
# Let's check if any account downgraded.
print(account_upgrades[account_upgrades['max'] > account_upgrades['last']])

            min  max  first  last  is_upgraded
account_id                                    
A01           1    2      1     1        False
A02           1    2      1     1        False
A03           1    2      1     1        False
A04           1    2      1     1        False
A06           2    3      2     2        False
A07           2    3      2     2        False
A08           2    3      2     2        False


In [7]:
# Let's check if any account upgraded but not from the first month.
print(account_upgrades[account_upgrades['max'] > account_upgrades['first']])

            min  max  first  last  is_upgraded
account_id                                    
A01           1    2      1     1        False
A02           1    2      1     1        False
A03           1    2      1     1        False
A04           1    2      1     1        False
A06           2    3      2     2        False
A07           2    3      2     2        False
A08           2    3      2     2        False


In [8]:
# The difference is 101.67.
# Let's double check the net invoice amount calculation.
# "net invoice amount" = amount - credit_amount
# Is there any other column?
print(invoices.columns)
print(invoices[['amount', 'credit_amount', 'net_amount']].head())

Index(['invoice_id', 'account_id', 'month', 'issue_date', 'due_date',
       'paid_date', 'plan_tier', 'amount', 'credit_amount', 'net_amount'],
      dtype='str')
   amount  credit_amount  net_amount
0     210              0         210
1     225              0         225
2     240              0         240
3     255              0         255
4     400              0         400


In [9]:
# Let's check if there are any missing values.
print(invoices.isnull().sum())
print(plan_history.isnull().sum())

invoice_id       0
account_id       0
month            0
issue_date       0
due_date         0
paid_date        0
plan_tier        0
amount           0
credit_amount    0
net_amount       0
dtype: int64
account_id    0
month         0
plan_tier     0
seats         0
tier_num      0
dtype: int64
